# FAseg — Batch Fine-tuning on In-Vivo Data (1 run)

Fine-tunes the **pretraining (Full) × best_s3 checkpoint** on in-vivo LPP
thigh data (`manual_seg_data`).  Click "Run All".

**`best_s3`** = the best pretraining checkpoint in **Stage 3**, i.e. the
lowest validation loss among epochs **61–100** (n1=20 + n2=40 → Stage 3 spans
epochs 61–100).

| Pretraining | Strategy | Output dir |
|-------------|----------|------------|
| pretraining (Full) | best_s3 | `outputs/pretraining/finetuning_best_s3/` |

Each run saves:
- `finetune_latest.pth` / `finetune_epochXX.pth` — all checkpoints
- `finetune_best.pth` — best fine-tuning epoch by val loss
- `finetune_loss_log.txt` — per-epoch train/val loss
- `source_info.txt` — which pretraining epoch was used


In [1]:
import os, sys, json, time, shutil
import torch
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import autocast, GradScaler

_NB_DIR = os.path.dirname(os.path.abspath(''))
if os.path.basename(_NB_DIR) == 'scripts':
    _project_root = os.path.dirname(_NB_DIR)
else:
    _project_root = _NB_DIR
_src_root = os.path.join(_project_root, 'src')
if _src_root not in sys.path:
    sys.path.insert(0, _src_root)

from faseg.data.dataset import RealLimbDataset
from faseg.models import UNet, pixelwise_cross_entropy_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [2]:
# =========================================================================
# Config — pretraining experiment + strategy to run
# =========================================================================
DATA_DIR = os.path.join(_project_root, 'manual segmentation', 'manual_seg_limb')

# (pretrain_dir, label) — only the Full pretraining, best_s3 checkpoint
PRETRAIN_EXPERIMENTS = [
    ('outputs/pretraining', 'Full'),
]

# Strategy:  (strategy_name,  output_subdir)
STRATEGIES = [
    ('best_s3',  'finetuning_best_s3'),
]

# Fine-tuning hyperparameters
FT_EPOCHS         = 50
FT_LR             = 5e-5
FT_BATCH          = 16
FT_VAL_RATIO      = 0.1
FT_NUM_WORKERS    = 8
FREEZE_ENCODER    = True
SCHEDULER_FACTOR  = 0.5
SCHEDULER_PATIENCE = 5

total_runs = len(PRETRAIN_EXPERIMENTS) * len(STRATEGIES)
print(f'Will run {total_runs} fine-tuning experiments')


Will run 1 fine-tuning experiments


In [3]:
def select_pretrain_checkpoint(pretrain_dir, strategy):
    """Return (epoch, ckpt_path, config_path)."""
    config_path = os.path.join(pretrain_dir, 'config.txt')

    if strategy == 'latest':
        ckpt = os.path.join(pretrain_dir, 'unet_latest.pth')
        if os.path.exists(ckpt):
            raw = torch.load(ckpt, map_location='cpu', weights_only=False)
            ep = raw.get('epoch', '?')
            return ep, ckpt, config_path

    # best_s3 (or fallback)
    loss_log = os.path.join(pretrain_dir, 'loss_log.txt')
    best_ep, best_val = None, float('inf')
    with open(loss_log) as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                ep, val = int(parts[0]), float(parts[2])
                if ep >= 61 and val < best_val:
                    best_val = val
                    best_ep = ep
    ckpt = os.path.join(pretrain_dir, f'unet_epoch{best_ep:02d}.pth')
    return best_ep, ckpt, config_path


def read_model_config(config_path):
    """Return (base_ch, dropout, use_bn) from pretraining config.txt."""
    if os.path.exists(config_path):
        with open(config_path) as f:
            cfg = json.load(f)
        return cfg.get('base_channel', 64), cfg.get('dropout_prob', 0.2), cfg.get('use_bn', True)
    return 64, 0.2, True


def run_one_finetune(pretrain_dir, label, strategy, output_subdir):
    """Run a single fine-tuning experiment.  Returns True on success."""
    pretrain_dir = os.path.join(_project_root, pretrain_dir)
    save_dir = os.path.join(pretrain_dir, output_subdir)

    print(f'\n{"="*60}')
    print(f'  {label}  |  strategy = {strategy}')
    print(f'  Pretrain : {pretrain_dir}')
    print(f'  Output   : {save_dir}')
    print(f'{"="*60}')

    # ---- Select pretrain checkpoint ----
    try:
        pr_epoch, ckpt_path, config_path = select_pretrain_checkpoint(pretrain_dir, strategy)
    except Exception as e:
        print(f'  !! SKIP: cannot select checkpoint — {e}')
        return False

    print(f'  Pretrain epoch : {pr_epoch}')
    print(f'  Checkpoint     : {ckpt_path}')

    # ---- Build model ----
    base_ch, dropout, use_bn = read_model_config(config_path)
    model = UNet(in_ch=1, base_ch=base_ch, num_classes=2,
                 dropout_prob=dropout, use_bn=use_bn).to(device)

    raw = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(raw, dict) and 'model_state_dict' in raw:
        state = raw['model_state_dict']          # full checkpoint
    else:
        state = raw                               # pure state_dict
    model.load_state_dict(state)

    # ---- Freeze encoder ----
    if FREEZE_ENCODER:
        for name, param in model.named_parameters():
            if name.startswith(('enc1', 'enc2', 'enc3', 'pool1', 'pool2', 'pool3')):
                param.requires_grad = False

    # ---- Dataset ----
    dataset = RealLimbDataset(DATA_DIR)
    val_size = int(len(dataset) * FT_VAL_RATIO)
    train_set, val_set = random_split(dataset, [len(dataset) - val_size, val_size])
    train_loader = DataLoader(train_set, batch_size=FT_BATCH, shuffle=True,
                              num_workers=FT_NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=FT_BATCH, shuffle=False,
                              num_workers=FT_NUM_WORKERS, pin_memory=True)

    # ---- Optimizer ----
    optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=FT_LR)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=SCHEDULER_FACTOR,
                                  patience=SCHEDULER_PATIENCE, verbose=False)
    scaler = GradScaler()

    # ---- Train ----
    os.makedirs(save_dir, exist_ok=True)
    loss_log_path = os.path.join(save_dir, 'finetune_loss_log.txt')
    best_ft_epoch, best_ft_val = 0, float('inf')

    for epoch in range(1, FT_EPOCHS + 1):
        t0 = time.time()
        model.train()
        tr_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            with autocast(device_type='cuda'):
                logits = model(imgs)
                loss = pixelwise_cross_entropy_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            tr_loss += loss.item() * imgs.size(0)
        tr_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                logits = model(imgs)
                val_loss += pixelwise_cross_entropy_loss(logits, masks).item() * imgs.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)
        elapsed = time.time() - t0

        is_best = val_loss < best_ft_val
        if is_best:
            best_ft_val, best_ft_epoch = val_loss, epoch

        with open(loss_log_path, 'a') as f:
            f.write(f'{epoch},{tr_loss:.6f},{val_loss:.6f}\n')

        ckpt = {'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'scheduler_state_dict': scheduler.state_dict()}
        torch.save(ckpt, os.path.join(save_dir, 'finetune_latest.pth'))
        torch.save(model.state_dict(), os.path.join(save_dir, f'finetune_epoch{epoch:02d}.pth'))

        marker = ' *' if is_best else ''
        print(f'  [FT {epoch:2d}/{FT_EPOCHS}]  tr={tr_loss:.4f}  val={val_loss:.4f}  {elapsed:.0f}s{marker}')

    # ---- Save best + source info ----
    best_src = os.path.join(save_dir, f'finetune_epoch{best_ft_epoch:02d}.pth')
    best_dst = os.path.join(save_dir, 'finetune_best.pth')
    if os.path.exists(best_src):
        shutil.copy2(best_src, best_dst)

    with open(os.path.join(save_dir, 'source_info.txt'), 'w') as f:
        f.write(f'pretrain_dir={pretrain_dir}\n')
        f.write(f'pretrain_epoch={pr_epoch}\n')
        f.write(f'strategy={strategy}\n')
        f.write(f'best_ft_epoch={best_ft_epoch}\n')
        f.write(f'best_ft_val={best_ft_val:.6f}\n')

    print(f'  DONE — best FT epoch={best_ft_epoch} (val={best_ft_val:.6f})')
    return True

In [4]:
# =========================================================================
# Run all
# =========================================================================
results = []
t_start = time.time()

for pretrain_dir, label in PRETRAIN_EXPERIMENTS:
    for strategy, output_subdir in STRATEGIES:
        ok = run_one_finetune(pretrain_dir, label, strategy, output_subdir)
        results.append((label, strategy, ok))

# =========================================================================
# Summary
# =========================================================================
print(f'\n{"="*60}')
print(f'  SUMMARY  (total time: {(time.time()-t_start)/60:.0f} min)')
print(f'{"="*60}')
for label, strategy, ok in results:
    status = 'OK' if ok else 'FAIL'
    print(f'  {label:20s}  {strategy:8s}  →  {status}')
print(f'\n  {sum(1 for _,_,ok in results if ok)} / {len(results)} completed')


  Full  |  strategy = best_s3
  Pretrain : /data/projects/AgentWork/FAseg for github/outputs/pretraining
  Output   : /data/projects/AgentWork/FAseg for github/outputs/pretraining/finetuning_best_s3
  Pretrain epoch : 84
  Checkpoint     : /data/projects/AgentWork/FAseg for github/outputs/pretraining/unet_epoch84.pth


/home/yifei-sun/anaconda3/envs/faseg_ablation/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(
/home/yifei-sun/anaconda3/envs/faseg_ablation/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  [FT  1/50]  tr=0.0626  val=0.0488  6s *
  [FT  2/50]  tr=0.0200  val=0.0245  5s *
  [FT  3/50]  tr=0.0167  val=0.0213  5s *
  [FT  4/50]  tr=0.0155  val=0.0202  5s *
  [FT  5/50]  tr=0.0146  val=0.0198  5s *
  [FT  6/50]  tr=0.0142  val=0.0194  5s *
  [FT  7/50]  tr=0.0139  val=0.0190  5s *
  [FT  8/50]  tr=0.0139  val=0.0188  5s *
  [FT  9/50]  tr=0.0136  val=0.0180  5s *
  [FT 10/50]  tr=0.0131  val=0.0179  5s *
  [FT 11/50]  tr=0.0128  val=0.0179  5s *
  [FT 12/50]  tr=0.0126  val=0.0177  5s *
  [FT 13/50]  tr=0.0124  val=0.0179  5s
  [FT 14/50]  tr=0.0122  val=0.0173  5s *
  [FT 15/50]  tr=0.0121  val=0.0174  5s
  [FT 16/50]  tr=0.0120  val=0.0173  5s
  [FT 17/50]  tr=0.0118  val=0.0171  5s *
  [FT 18/50]  tr=0.0118  val=0.0174  5s
  [FT 19/50]  tr=0.0114  val=0.0175  5s
  [FT 20/50]  tr=0.0114  val=0.0172  5s
  [FT 21/50]  tr=0.0113  val=0.0173  5s
  [FT 22/50]  tr=0.0112  val=0.0176  5s
  [FT 23/50]  tr=0.0113  val=0.0170  5s *
  [FT 24/50]  tr=0.0108  val=0.0173  5s
  [FT 25/5